In [1]:
#------------------------------------#
#------------- packages--------------#
#------------------------------------#

import pandas as pd
import numpy as np
from datetime import datetime, timedelta, date

import matplotlib.pyplot as plt
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.figure_factory as ff

import seaborn as sns
# from sklearn.preprocessing import MinMaxScaler

# show multiple outputs in a cell
# from IPython.core.interactiveshell import InteractiveShell
# InteractiveShell.ast_node_interactivity = "last_expr"
import requests
import os
from os import read

import rpy2
# import folium

In [2]:
# read in reverse geocoded participant data
locations = pd.read_csv('cville_revGeo.csv',parse_dates=['timestamp'])
# read in sleep and survey data
fitsurveys = pd.read_csv('slpsteps.csv',parse_dates=['date'])
# rename pid to ParticipantID
fitsurveys.rename(columns={'pid':'ParticipantID'},inplace=True)

## Aggregate location data

In [3]:
locations.head()

,ParticipantID,Work,timestamp,double_latitude,double_longitude,geometry,countyfp,name,namelsad,lsad,intptlat,intptlon,shape_leng,shape_area,address
0,vxx,Link Lab,2022-05-06 14:30:39,38.032120,-78.510380,POINT (-78.51038025044301 38.03211992498),51003,Albemarle,Albemarle County,6,38.024184,-78.553506,2.35713,3.035450e+09,"Olsson Hall, 151, Engineer's Way, McCormick Ro..."
1,vxx,Link Lab,2022-05-06 14:33:39,38.032120,-78.510380,POINT (-78.51038025044301 38.03211992498),51003,Albemarle,Albemarle County,6,38.024184,-78.553506,2.35713,3.035450e+09,"Olsson Hall, 151, Engineer's Way, McCormick Ro..."
2,vxx,Link Lab,2022-05-06 14:36:39,38.031862,-78.510286,POINT (-78.51028617938501 38.031861514449),51003,Albemarle,Albemarle County,6,38.024184,-78.553506,2.35713,3.035450e+09,"Olsson Hall, 151, Engineer's Way, McCormick Ro..."
3,vxx,Link Lab,2022-05-06 14:39:39,38.031863,-78.510284,POINT (-78.510284241017 38.031863054569),51003,Albemarle,Albemarle County,6,38.024184,-78.553506,2.35713,3.035450e+09,"Olsson Hall, 151, Engineer's Way, McCormick Ro..."
4,vxx,Link Lab,2022-05-06 14:42:39,38.031817,-78.510353,POINT (-78.510353091846 38.031817005834),51003,Albemarle,Albemarle County,6,38.024184,-78.553506,2.35713,3.035450e+09,"Olsson Hall, 151, Engineer's Way, McCormick Ro..."


In [4]:
# get unique participants
locations['ParticipantID'].unique()

array(['vxx', 'oyb', 'edr', 'xil', 'h9u', 'j02', 'pgm', 'lpz', 'uja',
       'mr1', 'heh', 'egl', 'ssg'], dtype=object)

In [5]:
# aggregate locations by date and participant
# chunk location data
pids = locations["ParticipantID"].unique()

locs = []

for p in pids:
    temp = locations[locations["ParticipantID"]==p]
    temp = temp.sort_values(by='timestamp')
    # if the address is the same as the next row, the give same group number
    temp['group'] = (temp['address'] != temp['address'].shift()).cumsum()
    # concat temp to locs
    locs.append(temp)

locs = pd.concat(locs).reset_index(drop=True)
# locs['day_name'] = locs["timestamp"].dt.day_name()
locs.head()

,ParticipantID,Work,timestamp,double_latitude,double_longitude,geometry,countyfp,name,namelsad,lsad,intptlat,intptlon,shape_leng,shape_area,address,group
0,vxx,Link Lab,2022-05-06 14:30:39,38.032120,-78.510380,POINT (-78.51038025044301 38.03211992498),51003,Albemarle,Albemarle County,6,38.024184,-78.553506,2.35713,3.035450e+09,"Olsson Hall, 151, Engineer's Way, McCormick Ro...",1
1,vxx,Link Lab,2022-05-06 14:33:39,38.032120,-78.510380,POINT (-78.51038025044301 38.03211992498),51003,Albemarle,Albemarle County,6,38.024184,-78.553506,2.35713,3.035450e+09,"Olsson Hall, 151, Engineer's Way, McCormick Ro...",1
2,vxx,Link Lab,2022-05-06 14:36:39,38.031862,-78.510286,POINT (-78.51028617938501 38.031861514449),51003,Albemarle,Albemarle County,6,38.024184,-78.553506,2.35713,3.035450e+09,"Olsson Hall, 151, Engineer's Way, McCormick Ro...",1
3,vxx,Link Lab,2022-05-06 14:39:39,38.031863,-78.510284,POINT (-78.510284241017 38.031863054569),51003,Albemarle,Albemarle County,6,38.024184,-78.553506,2.35713,3.035450e+09,"Olsson Hall, 151, Engineer's Way, McCormick Ro...",1
4,vxx,Link Lab,2022-05-06 14:42:39,38.031817,-78.510353,POINT (-78.510353091846 38.031817005834),51003,Albemarle,Albemarle County,6,38.024184,-78.553506,2.35713,3.035450e+09,"Olsson Hall, 151, Engineer's Way, McCormick Ro...",1


In [6]:
locs.to_csv('locs.csv')

In [7]:
locsagg = locs.groupby(['ParticipantID','group','address']).agg({'timestamp':['min','max'], 'double_latitude':'mean','double_longitude':'mean'}).reset_index()
locsagg.head()

ParticipantID group                                            address  \
                                                                           
0           edr     1  Olsson Hall, 151, Engineer's Way, McCormick Ro...   
1           edr     2  T3 Permit, Stadium Road, Jefferson Park Avenue...   
2           edr     3  Olsson Hall, 151, Engineer's Way, McCormick Ro...   
3           edr     4  T3 Permit, Stadium Road, Jefferson Park Avenue...   
4           edr     5  Olsson Hall, 151, Engineer's Way, McCormick Ro...   

            timestamp                     double_latitude double_longitude  
                  min                 max            mean             mean  
0 2022-04-26 14:53:23 2022-04-26 15:44:23       38.032043       -78.510271  
1 2022-04-26 15:47:23 2022-04-26 15:47:23       38.032052       -78.510098  
2 2022-04-26 15:50:23 2022-04-26 15:56:23       38.032026       -78.510213  
3 2022-04-26 15:59:23 2022-04-26 15:59:23       38.032021       -78.510052  
4 2022-04-26 16:02:23 2022-04-26 16:02:23       38.032002       -78.510154

In [8]:
locsagg.columns = ['ParticipantID','group','address','time_start','time_end','latitude','longitude']
# cast to datetime
locsagg['time_start'] = pd.to_datetime(locsagg['time_start'])
locsagg['time_end'] = pd.to_datetime(locsagg['time_end'])

locsagg.head()

,ParticipantID,group,address,time_start,time_end,latitude,longitude
0,edr,1,"Olsson Hall, 151, Engineer's Way, McCormick Ro...",2022-04-26 14:53:23,2022-04-26 15:44:23,38.032043,-78.510271
1,edr,2,"T3 Permit, Stadium Road, Jefferson Park Avenue...",2022-04-26 15:47:23,2022-04-26 15:47:23,38.032052,-78.510098
2,edr,3,"Olsson Hall, 151, Engineer's Way, McCormick Ro...",2022-04-26 15:50:23,2022-04-26 15:56:23,38.032026,-78.510213
3,edr,4,"T3 Permit, Stadium Road, Jefferson Park Avenue...",2022-04-26 15:59:23,2022-04-26 15:59:23,38.032021,-78.510052
4,edr,5,"Olsson Hall, 151, Engineer's Way, McCormick Ro...",2022-04-26 16:02:23,2022-04-26 16:02:23,38.032002,-78.510154


In [9]:
# how many observations per participant
locsagg.groupby('ParticipantID')['group'].count()

ParticipantID
edr     3935
egl    12371
h9u     5415
heh     6041
j02     1622
lpz      365
mr1     2008
oyb        9
pgm     1634
ssg     2391
uja     2469
vxx     1430
xil     1110
Name: group, dtype: int64

In [10]:
# drop oyb participant
locsagg = locsagg[locsagg['ParticipantID'] != 'oyb']
# how many observations per participant
locsagg.groupby('ParticipantID')['group'].count()

ParticipantID
edr     3935
egl    12371
h9u     5415
heh     6041
j02     1622
lpz      365
mr1     2008
pgm     1634
ssg     2391
uja     2469
vxx     1430
xil     1110
Name: group, dtype: int64

In [11]:
# get date
locsagg['date'] = locsagg['time_start'].dt.date
# get length of time spent at location
locsagg['time_spent'] = (locsagg['time_end'] - locsagg['time_start']).dt.total_seconds()
# reorganize the columns
locsagg = locsagg[['ParticipantID','group','date','time_start','time_end','time_spent','latitude','longitude','address']]
# create a diff time column time_start of next row - time_end of current row
locsagg['diff_time'] = locsagg['time_start'].shift(-1) - locsagg['time_end']

locsagg.head()

,ParticipantID,group,date,time_start,time_end,time_spent,latitude,longitude,address,diff_time
0,edr,1,2022-04-26,2022-04-26 14:53:23,2022-04-26 15:44:23,3060.0,38.032043,-78.510271,"Olsson Hall, 151, Engineer's Way, McCormick Ro...",0 days 00:03:00
1,edr,2,2022-04-26,2022-04-26 15:47:23,2022-04-26 15:47:23,0.0,38.032052,-78.510098,"T3 Permit, Stadium Road, Jefferson Park Avenue...",0 days 00:03:00
2,edr,3,2022-04-26,2022-04-26 15:50:23,2022-04-26 15:56:23,360.0,38.032026,-78.510213,"Olsson Hall, 151, Engineer's Way, McCormick Ro...",0 days 00:03:00
3,edr,4,2022-04-26,2022-04-26 15:59:23,2022-04-26 15:59:23,0.0,38.032021,-78.510052,"T3 Permit, Stadium Road, Jefferson Park Avenue...",0 days 00:03:00
4,edr,5,2022-04-26,2022-04-26 16:02:23,2022-04-26 16:02:23,0.0,38.032002,-78.510154,"Olsson Hall, 151, Engineer's Way, McCormick Ro...",0 days 00:03:00


In [12]:
# reset group
locsagg['group'] = np.nan
# Define the number of rows to check before and after
n_rows = 2


# Iterate over each participant
for participant in locsagg['ParticipantID'].unique():
    
    # Filter data for current participant
    grouped = locsagg.loc[locsagg['ParticipantID'] == participant]
    # Sort by date and time_start
    grouped = grouped.sort_values(['date', 'time_start'])
    # Start group number counter at 1 for each participant
    group_num = 0
    grouped['group'] = np.nan


    # Iterate over the data and assign group numbers
    for i, row in grouped.iterrows():
        # if its the first two rows, assign group number
        if i < n_rows:
            grouped.loc[i, 'group'] = group_num
            continue

        # Check if the current address is the same as two rows before
        current_address = row['address']
        # previous is two rows before the current
        previous_addresses = grouped.loc[max(i-n_rows, 0):i-1, 'address'].tolist()
        if (current_address in previous_addresses):
            # check the group number two rows before
            previous_group = grouped.loc[i-n_rows, 'group']
            # assign the same group number to the current row and row in beteen
            grouped.loc[i, 'group'] = previous_group
            grouped.loc[i-1, 'group'] = previous_group
        else:
            group_num += 1
            grouped.loc[i, 'group'] = group_num

    # Update the original dataframe with the new group numbers
    locsagg.loc[locsagg['ParticipantID'] == participant, 'group'] = grouped['group']

# Convert group numbers back to integer type
locsagg['group'] = locsagg['group'].astype(int)


In [13]:
# for each participant, every time there is a new number in group column, then increment new group number
for p in locsagg['ParticipantID'].unique():
    temp = locsagg[locsagg['ParticipantID']==p]
    temp['new_group'] = (temp['group'] != temp['group'].shift()).cumsum()
    locsagg.loc[locsagg['ParticipantID']==p,'new_group'] = temp['new_group']

# replace group number with new group number
locsagg['group'] = locsagg['new_group']
# drop new group column
locsagg.drop(columns=['new_group'], inplace=True)

In [14]:
# create a address2 column that records the second address in the group
locsagg['address2'] = locsagg.groupby(['ParticipantID', 'group'])['address'].shift(-1)
locsagg.head()

,ParticipantID,group,date,time_start,time_end,time_spent,latitude,longitude,address,diff_time,address2
0,edr,1.0,2022-04-26,2022-04-26 14:53:23,2022-04-26 15:44:23,3060.0,38.032043,-78.510271,"Olsson Hall, 151, Engineer's Way, McCormick Ro...",0 days 00:03:00,"T3 Permit, Stadium Road, Jefferson Park Avenue..."
1,edr,1.0,2022-04-26,2022-04-26 15:47:23,2022-04-26 15:47:23,0.0,38.032052,-78.510098,"T3 Permit, Stadium Road, Jefferson Park Avenue...",0 days 00:03:00,"Olsson Hall, 151, Engineer's Way, McCormick Ro..."
2,edr,1.0,2022-04-26,2022-04-26 15:50:23,2022-04-26 15:56:23,360.0,38.032026,-78.510213,"Olsson Hall, 151, Engineer's Way, McCormick Ro...",0 days 00:03:00,"T3 Permit, Stadium Road, Jefferson Park Avenue..."
3,edr,1.0,2022-04-26,2022-04-26 15:59:23,2022-04-26 15:59:23,0.0,38.032021,-78.510052,"T3 Permit, Stadium Road, Jefferson Park Avenue...",0 days 00:03:00,"Olsson Hall, 151, Engineer's Way, McCormick Ro..."
4,edr,1.0,2022-04-26,2022-04-26 16:02:23,2022-04-26 16:02:23,0.0,38.032002,-78.510154,"Olsson Hall, 151, Engineer's Way, McCormick Ro...",0 days 00:03:00,"T3 Permit, Stadium Road, Jefferson Park Avenue..."


In [15]:
# create a new dataframe 
locsagg2 = locsagg.copy()
# for each group, get the first time_start and last time_end and the address and address2
locsagg2 = locsagg2.groupby(['ParticipantID','group']).agg({'date':'first','time_start':'first','time_end':'last','latitude':'mean','longitude':'mean','address':'first','address2':'first'}).reset_index()
# get the length of time spent at location
locsagg2['time_spent'] = (locsagg2['time_end'] - locsagg2['time_start']).dt.total_seconds()
# reorder columns
locsagg2 = locsagg2[['ParticipantID','group','date','time_start','time_end','time_spent', 'latitude','longitude','address','address2']]
# group to int
locsagg2['group'] = locsagg2['group'].astype(int)
locsagg2.head()

,ParticipantID,group,date,time_start,time_end,time_spent,latitude,longitude,address,address2
0,edr,1,2022-04-26,2022-04-26 14:53:23,2022-04-26 16:08:23,4500.0,38.032036,-78.510156,"Olsson Hall, 151, Engineer's Way, McCormick Ro...","T3 Permit, Stadium Road, Jefferson Park Avenue..."
1,edr,2,2022-04-26,2022-04-26 16:11:23,2022-04-26 16:11:23,0.0,38.031746,-78.507757,"Jefferson Park Avenue, McCormick Road Houses, ...",None
2,edr,3,2022-04-26,2022-04-26 16:14:23,2022-04-26 16:14:23,0.0,38.031926,-78.505661,"South Lawn Commons Building, Jefferson Park Av...",None
3,edr,4,2022-04-26,2022-04-26 16:17:23,2022-04-26 16:17:23,0.0,38.032845,-78.503007,"Varsity Hall, Hospital Drive, Carr's Hill, Alb...",None
4,edr,5,2022-04-26,2022-04-26 16:20:23,2022-04-26 16:20:23,0.0,38.034504,-78.501330,"Hospital Drive, Carr's Hill, Albemarle County,...",None


In [16]:
[locsagg2.at[0, 'address'], locsagg2.at[0, 'address2']]

["Olsson Hall, 151, Engineer's Way, McCormick Road Houses, Charlottesville, Albemarle County, Virginia, 22903, United States",
 'T3 Permit, Stadium Road, Jefferson Park Avenue, McCormick Road Houses, Albemarle County, Virginia, 22903, United States']

In [17]:
# how many observations per participant
locsagg2.groupby('ParticipantID')['group'].count()

ParticipantID
edr    1959
egl    4103
h9u    1813
heh    4736
j02     878
lpz     276
mr1    1246
pgm    1145
ssg    1393
uja    1421
vxx     693
xil     767
Name: group, dtype: int64

In [18]:
# exclude observations where the time spent is 0
locsagg2 = locsagg2[locsagg2['time_spent']!=0]
# how many observations per participant
locsagg2.groupby('ParticipantID')['group'].count()

ParticipantID
edr     560
egl    1854
h9u     414
heh    1170
j02     236
lpz      90
mr1     322
pgm     339
ssg     383
uja     448
vxx     159
xil     273
Name: group, dtype: int64

In [19]:
# for each participant
# get the previous address and previous address2 in a list and check if the current address is in the list
# if the current address is in the previous_addresses list, then give the same group number
# if not, then increment group number
for p in locsagg2['ParticipantID'].unique():
    # Filter data for current participant
    grouped = locsagg2.loc[locsagg2['ParticipantID'] == p]
    # Sort by date and time_start
    grouped = grouped.sort_values(['date', 'time_start'])
    # Start group number counter at 1 for each participant
    group_num = 1
    # Create an empty list to store group numbers
    group_list = []
    # Assign initial group number
    group_list.append(group_num)

    # Iterate over the data for the current participant and assign group numbers
    for i in range(1, len(grouped)):
        # Get current and previous addresses and address2s
        current_address = grouped.iloc[i]['address']
        current_address2 = grouped.iloc[i]['address2']

        previous_address = grouped.iloc[i-1]['address']
        previous_address2 = grouped.iloc[i-1]['address2']

        # Check if current address matches the previous address or address2
        if ((current_address == previous_address or current_address == previous_address2) or
                (current_address2 == previous_address or current_address2 == previous_address2) and
                current_address2 is not None):
            # Assign the same group number as the previous row
            group_list.append(group_list[-1])
        else:
            # Increment the group number
            group_num += 1
            group_list.append(group_num)

    # Update the original dataframe with the new group numbers for the current participant
    locsagg2.loc[locsagg2['ParticipantID'] == p, 'group'] = group_list


# Convert group numbers back to integer type
# locsagg2['group'] = locsagg2['group'].astype(int)


In [20]:
# Concatenate 'address' and 'address2' columns, exclude None/NaN values
locsagg2['list_address'] = locsagg2.apply(lambda x: list(set([x['address'], x['address2']])), axis=1)
locsagg2['list_address'] = locsagg2['list_address'].apply(lambda x: [addr for addr in x if pd.notna(addr)])
locsagg2.head()

,ParticipantID,group,date,time_start,time_end,time_spent,latitude,longitude,address,address2,list_address
0,edr,1,2022-04-26,2022-04-26 14:53:23,2022-04-26 16:08:23,4500.0,38.032036,-78.510156,"Olsson Hall, 151, Engineer's Way, McCormick Ro...","T3 Permit, Stadium Road, Jefferson Park Avenue...","[T3 Permit, Stadium Road, Jefferson Park Avenu..."
6,edr,2,2022-04-26,2022-04-26 16:26:23,2022-04-26 17:02:23,2160.0,38.033924,-78.499180,"1327,1329, West Main Street, Venable, Carr's H...","1331, West Main Street, Venable, Carr's Hill, ...","[1331, West Main Street, Venable, Carr's Hill,..."
11,edr,3,2022-04-26,2022-04-26 17:17:23,2022-04-26 21:57:27,16804.0,38.032117,-78.510272,"T3 Permit, Stadium Road, Jefferson Park Avenue...","Olsson Hall, 151, Engineer's Way, McCormick Ro...","[T3 Permit, Stadium Road, Jefferson Park Avenu..."
12,edr,3,2022-04-26,2022-04-26 22:00:27,2022-04-26 23:00:27,3600.0,38.032120,-78.510239,"T3 Permit, Stadium Road, Jefferson Park Avenue...","Olsson Hall, 151, Engineer's Way, McCormick Ro...","[T3 Permit, Stadium Road, Jefferson Park Avenu..."
13,edr,3,2022-04-26,2022-04-26 23:03:27,2022-04-27 01:27:27,8640.0,38.032172,-78.510254,"Thornton Hall, Woodrow Street, Jefferson Park ...","Olsson Hall, 151, Engineer's Way, McCormick Ro...","[Olsson Hall, 151, Engineer's Way, McCormick R..."


In [21]:
# Define custom function to get unique addresses
def unique_addresses(x):
    all_addresses = [addr for sublist in x for addr in sublist if pd.notna(addr)]
    return list(set(all_addresses))

In [22]:
# for each group, get the first time_start and last time_end and the address and address2
locsunique = locsagg2.groupby(['ParticipantID', 'group']).agg(
    date=('date', 'first'),
    time_start=('time_start', 'first'),
    time_end=('time_end', 'last'),
    latitude=('latitude', 'mean'),
    longitude=('longitude', 'mean'),
    unique_address=('list_address', unique_addresses) # Use custom aggregation function
).reset_index()
# get the length of time spent at location
locsunique['time_spent'] = (locsunique['time_end'] - locsunique['time_start']).dt.total_seconds()
# reorder columns
locsunique = locsunique[['ParticipantID','group','date','time_start','time_end','time_spent', 'latitude','longitude','unique_address']]
locsunique.head()

,ParticipantID,group,date,time_start,time_end,time_spent,latitude,longitude,unique_address
0,edr,1,2022-04-26,2022-04-26 14:53:23,2022-04-26 16:08:23,4500.0,38.032036,-78.510156,"[T3 Permit, Stadium Road, Jefferson Park Avenu..."
1,edr,2,2022-04-26,2022-04-26 16:26:23,2022-04-26 17:02:23,2160.0,38.033924,-78.499180,"[1331, West Main Street, Venable, Carr's Hill,..."
2,edr,3,2022-04-26,2022-04-26 17:17:23,2022-04-27 02:03:27,31564.0,38.032121,-78.510238,"[T3 Permit, Stadium Road, Jefferson Park Avenu..."
3,edr,4,2022-04-27,2022-04-27 02:18:27,2022-04-27 12:03:28,35101.0,38.047333,-78.516904,"[Crestwood Drive, University Heights, Albemarl..."
4,edr,5,2022-04-27,2022-04-27 12:15:28,2022-04-27 22:39:28,37440.0,38.032005,-78.510192,"[T3 Permit, Stadium Road, Jefferson Park Avenu..."


#### Let's reverse geocode averaged coordinates for comparison

In [23]:
# # reverse geocode using Nominatim API and requests package
# def get_reverse(lat, lon):
#     url = f'http://localhost:8080/reverse?lat={lat}&lon={lon}&format=json'
#     try:
#         result = requests.get(url)
#         result_json = result.json()
#         return result_json['display_name']
#     except:
#         return None

In [24]:
# # reverse geocode latitude and longitude
# locsunique['revGeo_address'] = locsunique.apply(lambda x: get_reverse(x['latitude'], x['longitude']), axis=1)
# # to csv 
# locsunique.to_csv('locsunique.csv', index=False)
# read csv
locsunique = pd.read_csv('locsunique.csv',parse_dates=['date','time_start','time_end'])
locsunique.head()

,ParticipantID,group,date,time_start,time_end,time_spent,latitude,longitude,unique_address,revGeo_address
0,edr,1,2022-04-26,2022-04-26 14:53:00,2022-04-26 16:08:00,4500,38.032036,-78.510156,"[""Olsson Hall, 151, Engineer's Way, McCormick ...","Olsson Hall, 151, Engineer's Way, McCormick Ro..."
1,edr,2,2022-04-26,2022-04-26 16:26:00,2022-04-26 17:02:00,2160,38.033924,-78.499180,"[""1327,1329, West Main Street, Venable, Carr's...","1327,1329, West Main Street, Venable, Carr's H..."
2,edr,3,2022-04-26,2022-04-26 17:17:00,2022-04-26 23:59:00,24157,38.032121,-78.510238,"['Thornton Hall, Woodrow Street, Jefferson Par...","Olsson Hall, 151, Engineer's Way, McCormick Ro..."
3,edr,3,2022-04-27,2022-04-27 00:00:00,2022-04-27 02:03:00,7407,38.032121,-78.510238,"['Thornton Hall, Woodrow Street, Jefferson Par...","Olsson Hall, 151, Engineer's Way, McCormick Ro..."
4,edr,4,2022-04-27,2022-04-27 02:18:00,2022-04-27 12:03:00,35101,38.047333,-78.516904,"['Crestwood Drive, University Heights, Albemar...","Crestwood Drive, University Heights, Albemarle..."


In [25]:
# create a new column check address 
# true if revGeo_address is in unique_address
# false if revGeo_address is not in unique_address
locsunique['check_address'] = locsunique.apply(lambda x: x['revGeo_address'] in x['unique_address'], axis=1)
# check how many False
locsunique['check_address'].value_counts()

True     5182
False     438
Name: check_address, dtype: int64

In [26]:
# how many observations per participant
locsunique.groupby('ParticipantID')['group'].count()

ParticipantID
edr     566
egl    1153
h9u     413
heh    1168
j02     239
lpz     103
mr1     308
pgm     367
ssg     396
uja     477
vxx     162
xil     268
Name: group, dtype: int64

In [27]:
# create column at_Eschool
# true if list_address contains the partial string "Olsson Hall" or "Thornton Hall" or "Rice Hall"
# Create a boolean mask for addresses that contain any of the partial strings
partial_strings = ["Olsson Hall", "Thornton Hall", "Rice Hall","T3 Permit"]
mask = locsunique['unique_address'].apply(lambda x: any([s in x for s in partial_strings]))
# Assign True to rows that match the mask, False otherwise
locsunique['at_Eschool'] = mask
locsunique.head()

,ParticipantID,group,date,time_start,time_end,time_spent,latitude,longitude,unique_address,revGeo_address,check_address,at_Eschool
0,edr,1,2022-04-26,2022-04-26 14:53:00,2022-04-26 16:08:00,4500,38.032036,-78.510156,"[""Olsson Hall, 151, Engineer's Way, McCormick ...","Olsson Hall, 151, Engineer's Way, McCormick Ro...",True,True
1,edr,2,2022-04-26,2022-04-26 16:26:00,2022-04-26 17:02:00,2160,38.033924,-78.499180,"[""1327,1329, West Main Street, Venable, Carr's...","1327,1329, West Main Street, Venable, Carr's H...",True,False
2,edr,3,2022-04-26,2022-04-26 17:17:00,2022-04-26 23:59:00,24157,38.032121,-78.510238,"['Thornton Hall, Woodrow Street, Jefferson Par...","Olsson Hall, 151, Engineer's Way, McCormick Ro...",True,True
3,edr,3,2022-04-27,2022-04-27 00:00:00,2022-04-27 02:03:00,7407,38.032121,-78.510238,"['Thornton Hall, Woodrow Street, Jefferson Par...","Olsson Hall, 151, Engineer's Way, McCormick Ro...",True,True
4,edr,4,2022-04-27,2022-04-27 02:18:00,2022-04-27 12:03:00,35101,38.047333,-78.516904,"['Crestwood Drive, University Heights, Albemar...","Crestwood Drive, University Heights, Albemarle...",True,False


In [28]:
# time spent over 24 hours (86400 seconds) is not possible
locsunique[locsunique['time_spent']>86400]

,ParticipantID,group,date,time_start,time_end,time_spent,latitude,longitude,unique_address,revGeo_address,check_address,at_Eschool


In [29]:
def split_rows_by_day(df):
    """
    Splits rows in a pandas DataFrame by day when time_start and time_end span across multiple days.
    
    Args:
        df (pd.DataFrame): Input DataFrame with columns 'time_start', 'time_end', and 'time_spent' in datetime format.
        
    Returns:
        pd.DataFrame: DataFrame with rows split by day when time_start and time_end span across multiple days.
    """
    # Create a copy of the input DataFrame to avoid modifying the original data
    df_split = df.copy()
    
    # Identify rows where time_start and time_end span across multiple days
    multi_day_rows = df_split[df_split['time_start'].dt.date != df_split['time_end'].dt.date]
    
    # Iterate over multi-day rows
    for idx, row in multi_day_rows.iterrows():
        # Extract the date portion from the datetime objects
        date_start = row['time_start'].date()
        date_end = row['time_end'].date()

        # Calculate the start and end times for the first day
        end_time_day1 = datetime.combine(date_start, datetime.max.time())  # End of the day (23:59:59)
        start_time_day2 = datetime.combine(date_end, datetime.min.time())  # Beginning of the day (00:00:00) of the subsequent day

        # Calculate the time spent for the first day and the subsequent day
        time_spent_day1 = (end_time_day1 - row['time_start']).total_seconds()
        time_spent_day2 = (row['time_end'] - start_time_day2).total_seconds()

        # Round the time spent for the first day and the subsequent day to the nearest second
        time_spent_day1 = round(time_spent_day1)
        time_spent_day2 = round(time_spent_day2)

        # Update the 'time_end' and 'time_spent' for the first day in the original row
        df_split.at[idx, 'time_end'] = end_time_day1
        df_split.at[idx, 'time_spent'] = time_spent_day1

        # Create a new row for the subsequent day
        new_row = row.copy()
        new_row['date'] = date_end
        new_row['time_start'] = start_time_day2
        new_row['time_end'] = row['time_end']
        new_row['time_spent'] = time_spent_day2

        # Append the new row to the DataFrame
        df_split = df_split.append(new_row, ignore_index=True)

    # Sort the DataFrame by ParticipantID and time_start
    df_split = df_split.sort_values(['ParticipantID', 'time_start']).reset_index(drop=True)
    # Round down the time_end to the nearest second to avoid rounding up to the next day
    df_split['time_end'] = df_split['time_end'].dt.floor('1s')
    df_split['date'] = df_split['time_start'].dt.date

    return df_split

In [30]:
locsunique = split_rows_by_day(locsunique)
# if different dates
locsunique[locsunique['time_start'].dt.date !=locsunique['time_end'].dt.date]
# remove rows where time_spent = 0
# locsunique = locsunique[locsunique['time_spent']!=0]
# locsunique[locsunique['time_spent']>86400]

,ParticipantID,group,date,time_start,time_end,time_spent,latitude,longitude,unique_address,revGeo_address,check_address,at_Eschool


In [31]:
locsunique.head()

,ParticipantID,group,date,time_start,time_end,time_spent,latitude,longitude,unique_address,revGeo_address,check_address,at_Eschool
0,edr,1,2022-04-26,2022-04-26 14:53:00,2022-04-26 16:08:00,4500,38.032036,-78.510156,"[""Olsson Hall, 151, Engineer's Way, McCormick ...","Olsson Hall, 151, Engineer's Way, McCormick Ro...",True,True
1,edr,2,2022-04-26,2022-04-26 16:26:00,2022-04-26 17:02:00,2160,38.033924,-78.499180,"[""1327,1329, West Main Street, Venable, Carr's...","1327,1329, West Main Street, Venable, Carr's H...",True,False
2,edr,3,2022-04-26,2022-04-26 17:17:00,2022-04-26 23:59:00,24157,38.032121,-78.510238,"['Thornton Hall, Woodrow Street, Jefferson Par...","Olsson Hall, 151, Engineer's Way, McCormick Ro...",True,True
3,edr,3,2022-04-27,2022-04-27 00:00:00,2022-04-27 02:03:00,7407,38.032121,-78.510238,"['Thornton Hall, Woodrow Street, Jefferson Par...","Olsson Hall, 151, Engineer's Way, McCormick Ro...",True,True
4,edr,4,2022-04-27,2022-04-27 02:18:00,2022-04-27 12:03:00,35101,38.047333,-78.516904,"['Crestwood Drive, University Heights, Albemar...","Crestwood Drive, University Heights, Albemarle...",True,False


In [32]:
locsunique.shape

(5620, 12)

In [33]:
# Group by ParticipantID and revGeo address to get the total time spent at each location, get the coordinates too
freqLocs = locsunique.groupby(['ParticipantID','revGeo_address', ]).agg({'latitude': 'mean','longitude': 'mean','time_spent': 'sum'}).reset_index()
# sort by time spent
freqLocs = freqLocs.sort_values(['ParticipantID', 'time_spent'], ascending=[True, False])
# to csv
# freqLocs.to_csv('freqLocs.csv', index=False)
# want top 10 locations per participant
freqLocs = freqLocs.groupby('ParticipantID').head(3)
freqLocs

,ParticipantID,revGeo_address,latitude,longitude,time_spent
64,edr,"Crestwood Drive, University Heights, Albemarle...",38.047312,-78.516879,3712142
101,edr,"Olsson Hall, 151, Engineer's Way, McCormick Ro...",38.031995,-78.510329,1338140
82,edr,"Ivy Drive, University Heights, Albemarle Count...",38.047828,-78.515909,201057
167,egl,"2310, Price Avenue, Jefferson Park Avenue, Cha...",38.027092,-78.517568,3032071
169,egl,"2312, Price Avenue, Jefferson Park Avenue, Cha...",38.027039,-78.517673,908931
218,egl,"Olsson Hall, 151, Engineer's Way, McCormick Ro...",38.032050,-78.510418,534905
250,h9u,"100, Hurst Lane, Johnson Village, West View Te...",38.018285,-78.498709,2658500
327,h9u,"Old Trail Drive, Old Trail, Crozet, Albemarle ...",38.063391,-78.714422,545064
253,h9u,"104, Hurst Lane, Johnson Village, West View Te...",38.018184,-78.498834,469233
417,heh,"1821, Jefferson Park Avenue, Charlottesville, ...",38.027995,-78.511485,2592282


In [34]:
# create a dataframe home where it is the most frequent location for each participant, reset index
home = freqLocs.groupby('ParticipantID').first().reset_index()
# rename revGeo_address to home_address, home_latitude, home_longitude
home = home.rename(columns={'revGeo_address': 'home_address', 'latitude': 'home_latitude', 'longitude': 'home_longitude'})
# dont need time_spent
home = home.drop(columns=['time_spent'])
home

,ParticipantID,home_address,home_latitude,home_longitude
0,edr,"Crestwood Drive, University Heights, Albemarle...",38.047312,-78.516879
1,egl,"2310, Price Avenue, Jefferson Park Avenue, Cha...",38.027092,-78.517568
2,h9u,"100, Hurst Lane, Johnson Village, West View Te...",38.018285,-78.498709
3,heh,"1821, Jefferson Park Avenue, Charlottesville, ...",38.027995,-78.511485
4,j02,"446, Farrish Circle, Copeley Hill Apartments, ...",38.047713,-78.510204
5,lpz,"510, Seymour Road, Copeley Hill Apartments, Al...",38.049435,-78.510169
6,mr1,"10th & Page, Starr Hill, Charlottesville, Virg...",38.037550,-78.491532
7,pgm,"Willow Oak Circle, Ivy Oaks, Albemarle County,...",38.075970,-78.593104
8,ssg,"The Flats @ West Village, 852, West Main Stree...",38.031889,-78.493549
9,uja,"547, Seymour Road, Copeley Hill Apartments, Al...",38.050506,-78.510311


In [35]:
# merge home to locsunique on ParticipantID
locsunique = pd.merge(locsunique, home, on='ParticipantID', how='left')
locsunique.head()

,ParticipantID,group,date,time_start,time_end,time_spent,latitude,longitude,unique_address,revGeo_address,check_address,at_Eschool,home_address,home_latitude,home_longitude
0,edr,1,2022-04-26,2022-04-26 14:53:00,2022-04-26 16:08:00,4500,38.032036,-78.510156,"[""Olsson Hall, 151, Engineer's Way, McCormick ...","Olsson Hall, 151, Engineer's Way, McCormick Ro...",True,True,"Crestwood Drive, University Heights, Albemarle...",38.047312,-78.516879
1,edr,2,2022-04-26,2022-04-26 16:26:00,2022-04-26 17:02:00,2160,38.033924,-78.499180,"[""1327,1329, West Main Street, Venable, Carr's...","1327,1329, West Main Street, Venable, Carr's H...",True,False,"Crestwood Drive, University Heights, Albemarle...",38.047312,-78.516879
2,edr,3,2022-04-26,2022-04-26 17:17:00,2022-04-26 23:59:00,24157,38.032121,-78.510238,"['Thornton Hall, Woodrow Street, Jefferson Par...","Olsson Hall, 151, Engineer's Way, McCormick Ro...",True,True,"Crestwood Drive, University Heights, Albemarle...",38.047312,-78.516879
3,edr,3,2022-04-27,2022-04-27 00:00:00,2022-04-27 02:03:00,7407,38.032121,-78.510238,"['Thornton Hall, Woodrow Street, Jefferson Par...","Olsson Hall, 151, Engineer's Way, McCormick Ro...",True,True,"Crestwood Drive, University Heights, Albemarle...",38.047312,-78.516879
4,edr,4,2022-04-27,2022-04-27 02:18:00,2022-04-27 12:03:00,35101,38.047333,-78.516904,"['Crestwood Drive, University Heights, Albemar...","Crestwood Drive, University Heights, Albemarle...",True,False,"Crestwood Drive, University Heights, Albemarle...",38.047312,-78.516879


In [36]:
# in locsunique, create a column at_home where it is true if home_address the same as revGeo_address
locsunique['at_home'] = locsunique.apply(lambda x: x['home_address'] == x['revGeo_address'], axis=1)
# dont need home_latitude, home_longitude
locsunique = locsunique.drop(columns=['home_latitude', 'home_longitude'])
# create column at_else where it is true if at_home is false and at_Eschool is false
locsunique['at_else'] = locsunique.apply(lambda x: x['at_home'] == False and x['at_Eschool'] == False, axis=1)
locsunique.head()

,ParticipantID,group,date,time_start,time_end,time_spent,latitude,longitude,unique_address,revGeo_address,check_address,at_Eschool,home_address,at_home,at_else
0,edr,1,2022-04-26,2022-04-26 14:53:00,2022-04-26 16:08:00,4500,38.032036,-78.510156,"[""Olsson Hall, 151, Engineer's Way, McCormick ...","Olsson Hall, 151, Engineer's Way, McCormick Ro...",True,True,"Crestwood Drive, University Heights, Albemarle...",False,False
1,edr,2,2022-04-26,2022-04-26 16:26:00,2022-04-26 17:02:00,2160,38.033924,-78.499180,"[""1327,1329, West Main Street, Venable, Carr's...","1327,1329, West Main Street, Venable, Carr's H...",True,False,"Crestwood Drive, University Heights, Albemarle...",False,True
2,edr,3,2022-04-26,2022-04-26 17:17:00,2022-04-26 23:59:00,24157,38.032121,-78.510238,"['Thornton Hall, Woodrow Street, Jefferson Par...","Olsson Hall, 151, Engineer's Way, McCormick Ro...",True,True,"Crestwood Drive, University Heights, Albemarle...",False,False
3,edr,3,2022-04-27,2022-04-27 00:00:00,2022-04-27 02:03:00,7407,38.032121,-78.510238,"['Thornton Hall, Woodrow Street, Jefferson Par...","Olsson Hall, 151, Engineer's Way, McCormick Ro...",True,True,"Crestwood Drive, University Heights, Albemarle...",False,False
4,edr,4,2022-04-27,2022-04-27 02:18:00,2022-04-27 12:03:00,35101,38.047333,-78.516904,"['Crestwood Drive, University Heights, Albemar...","Crestwood Drive, University Heights, Albemarle...",True,False,"Crestwood Drive, University Heights, Albemarle...",True,False


In [37]:
# locsunique.to_csv('locations.csv', index=False)

In [38]:
# read in StudyParticipants.csv
pinfo = pd.read_csv('StudyParticipants.csv')
# only need where they Work
pinfo = pinfo[['ParticipantID', 'Work']]
# in Work, replace "Off Grounds BR" with "Off Grounds" bc we dont need to know whether workspace=BR yet
pinfo['Work'] = pinfo['Work'].replace('Off Grounds BR', 'Off Grounds')
pinfo

,ParticipantID,Work
0,edr,Link Lab
1,egl,Link Lab
2,j02,Off Grounds
3,uja,Link Lab
4,lpz,Off Grounds
5,oyb,Link Lab
6,mr1,Link Lab
7,xil,Link Lab
8,pgm,Link Lab
9,h9u,Off Grounds


In [39]:
# merge pinfo to locsunique on ParticipantID
locsunique = pd.merge(locsunique, pinfo, on='ParticipantID', how='left')
locsunique.head()

,ParticipantID,group,date,time_start,time_end,time_spent,latitude,longitude,unique_address,revGeo_address,check_address,at_Eschool,home_address,at_home,at_else,Work
0,edr,1,2022-04-26,2022-04-26 14:53:00,2022-04-26 16:08:00,4500,38.032036,-78.510156,"[""Olsson Hall, 151, Engineer's Way, McCormick ...","Olsson Hall, 151, Engineer's Way, McCormick Ro...",True,True,"Crestwood Drive, University Heights, Albemarle...",False,False,Link Lab
1,edr,2,2022-04-26,2022-04-26 16:26:00,2022-04-26 17:02:00,2160,38.033924,-78.499180,"[""1327,1329, West Main Street, Venable, Carr's...","1327,1329, West Main Street, Venable, Carr's H...",True,False,"Crestwood Drive, University Heights, Albemarle...",False,True,Link Lab
2,edr,3,2022-04-26,2022-04-26 17:17:00,2022-04-26 23:59:00,24157,38.032121,-78.510238,"['Thornton Hall, Woodrow Street, Jefferson Par...","Olsson Hall, 151, Engineer's Way, McCormick Ro...",True,True,"Crestwood Drive, University Heights, Albemarle...",False,False,Link Lab
3,edr,3,2022-04-27,2022-04-27 00:00:00,2022-04-27 02:03:00,7407,38.032121,-78.510238,"['Thornton Hall, Woodrow Street, Jefferson Par...","Olsson Hall, 151, Engineer's Way, McCormick Ro...",True,True,"Crestwood Drive, University Heights, Albemarle...",False,False,Link Lab
4,edr,4,2022-04-27,2022-04-27 02:18:00,2022-04-27 12:03:00,35101,38.047333,-78.516904,"['Crestwood Drive, University Heights, Albemar...","Crestwood Drive, University Heights, Albemarle...",True,False,"Crestwood Drive, University Heights, Albemarle...",True,False,Link Lab


### Lets get summary of where participant spends their day

In [40]:
# group by participant and date to get count of unqiue groups per day
grpsCount = locsunique.groupby(['ParticipantID', 'date']).agg(
    count_groups=('group', pd.Series.nunique),
).reset_index()
# convert date to datetime
grpsCount['date'] = pd.to_datetime(grpsCount['date'])
# get day of week
grpsCount['day_of_week'] = grpsCount['date'].dt.day_name()
# reorder columns
grpsCount = grpsCount[['ParticipantID', 'date', 'day_of_week', 'count_groups']]
grpsCount.head()

,ParticipantID,date,day_of_week,count_groups
0,edr,2022-04-26,Tuesday,3
1,edr,2022-04-27,Wednesday,4
2,edr,2022-04-28,Thursday,4
3,edr,2022-04-29,Friday,10
4,edr,2022-04-30,Saturday,4


In [41]:
# group by ParticipantID and date
# add the time spent elsewhere
atElse = locsunique[locsunique['at_else']==True]
atElse_day = atElse.groupby(['ParticipantID', 'date']).agg(
    time_else_sec=('time_spent', 'sum'),
).reset_index()
# convert date to datetime
atElse_day['date'] = pd.to_datetime(atElse_day['date'])
# get time spent in hours
atElse_day['time_else_hr'] = atElse_day['time_else_sec']/3600
atElse_day.head()

,ParticipantID,date,time_else_sec,time_else_hr
0,edr,2022-04-26,2160,0.600000
1,edr,2022-04-28,6299,1.749722
2,edr,2022-04-29,15301,4.250278
3,edr,2022-04-30,44020,12.227778
4,edr,2022-05-01,60203,16.723056


In [42]:
# group by ParticipantID and date
# add the time spent at Eschool
# create a dataframe to keep only the observations where at_Eschool is True
atEschool = locsunique[locsunique['at_Eschool']==True]
atEschool_day = atEschool.groupby(['ParticipantID', 'date']).agg(
    time_eschool_sec=('time_spent', 'sum')
).reset_index()
# convert date to datetime
atEschool_day['date'] = pd.to_datetime(atEschool_day['date'])
# compute the time spent in hours
atEschool_day['time_eschool_hr'] = atEschool_day['time_eschool_sec']/3600
atEschool_day.head()

,ParticipantID,date,time_eschool_sec,time_eschool_hr
0,edr,2022-04-26,28657,7.960278
1,edr,2022-04-27,44847,12.457500
2,edr,2022-04-28,27223,7.561944
3,edr,2022-04-29,7877,2.188056
4,edr,2022-05-01,10620,2.950000


In [43]:
# group by ParticipantID and date
# add the time spent at home
atHome = locsunique[locsunique['at_home']==True]
atHome_day = atHome.groupby(['ParticipantID', 'date']).agg(
    time_home_sec=('time_spent', 'sum')
).reset_index()
# convert date to datetime
atHome_day['date'] = pd.to_datetime(atHome_day['date'])
# compute the time spent in hours
atHome_day['time_home_hr'] = atHome_day['time_home_sec']/3600
atHome_day.head()

,ParticipantID,date,time_home_sec,time_home_hr
0,edr,2022-04-27,39213,10.892500
1,edr,2022-04-28,51438,14.288333
2,edr,2022-04-29,52194,14.498333
3,edr,2022-04-30,37259,10.349722
4,edr,2022-05-01,11257,3.126944


In [44]:
# merge the dataframes
day_sum = pd.merge(grpsCount, atElse_day, on=['ParticipantID', 'date'], how='left')
day_sum = pd.merge(day_sum, atEschool_day, on=['ParticipantID', 'date'], how='left')
day_sum = pd.merge(day_sum, atHome_day, on=['ParticipantID', 'date'], how='left')
# merge with pinfo
day_sum = pd.merge(day_sum, pinfo, on='ParticipantID', how='left')
# reorder columns
day_sum = day_sum[['ParticipantID', 'Work', 'date', 'day_of_week', 'count_groups', 'time_else_hr', 'time_eschool_hr', 'time_home_hr']]
# cast date to datetime
day_sum['date'] = pd.to_datetime(day_sum['date'])
day_sum.head()

,ParticipantID,Work,date,day_of_week,count_groups,time_else_hr,time_eschool_hr,time_home_hr
0,edr,Link Lab,2022-04-26,Tuesday,3,0.600000,7.960278,NaN
1,edr,Link Lab,2022-04-27,Wednesday,4,NaN,12.457500,10.892500
2,edr,Link Lab,2022-04-28,Thursday,4,1.749722,7.561944,14.288333
3,edr,Link Lab,2022-04-29,Friday,10,4.250278,2.188056,14.498333
4,edr,Link Lab,2022-04-30,Saturday,4,12.227778,NaN,10.349722


In [45]:
# how many null values in time_outside_hr and time_eschool_hr
# day_sum.isnull().sum()

# fill null values with 0
day_sum = day_sum.fillna(0)

In [46]:
# how many observations per participant
day_sum.groupby('ParticipantID')['date'].count()

ParticipantID
edr    80
egl    70
h9u    57
heh    57
j02    42
lpz    24
mr1    42
pgm    41
ssg    59
uja    67
vxx    34
xil    50
Name: date, dtype: int64

## Merge locations

In [47]:
# how many observations per participant in fitsurveys
fitsurveys.groupby('ParticipantID')['date'].count()

ParticipantID
edr    27
egl    44
h9u    58
heh    52
lpz    48
mr1    28
oyb    19
ssg    26
uja    33
xil    32
Name: date, dtype: int64

In [48]:
fitsurveys.columns

Index(['ParticipantID', 'group', 'wakeDate', 'day_name', 'bedtime', 'wakeup',
       'duration', 'TST', 'REM:nREM', 'deep', 'light', 'rem', 'sleep', 'Q4',
       'rep_bedtime', 'rep_wakeup', 'day_before', 'survey_index', 'EndDate',
       'activity', 'Q29', 'stress', 'productivity', 'Q27_1', 'Q34',
       'Q34_6_TEXT', 'Q35', 'date', 'steps', 'steps_norm', 'steps_std',
       'week'],
      dtype='object')

In [49]:
# merge location day_sum to fitsurveys on ParticipantID and date (day_sum) and WakeDate (fitsurveys)
# wakeDate to datetime
fitsurveys['wakeDate'] = pd.to_datetime(fitsurveys['wakeDate'])
merged = pd.merge(fitsurveys, day_sum, left_on=['ParticipantID', 'wakeDate'], right_on=['ParticipantID', 'date'], how='left')
merged.head()

,ParticipantID,group,wakeDate,day_name,bedtime,wakeup,duration,TST,REM:nREM,deep,...,steps_norm,steps_std,week,Work,date_y,day_of_week,count_groups,time_else_hr,time_eschool_hr,time_home_hr
0,edr,1,2022-04-28,Thursday,2022-04-28 01:51:00,2022-04-28 10:30:00,31140.0,7.633333,0.275766,6060,...,0.685709,1.226819,weekday,Link Lab,2022-04-28,Thursday,4.0,1.749722,7.561944,14.288333
1,edr,2,2022-04-29,Friday,2022-04-29 01:13:00,2022-04-29 09:07:30,28470.0,7.475000,0.321060,4110,...,0.075270,-1.145889,weekday,Link Lab,2022-04-29,Friday,10.0,4.250278,2.188056,14.498333
2,edr,5,2022-05-02,Monday,2022-05-02 00:59:00,2022-05-02 08:27:00,26880.0,6.916667,0.328000,5760,...,0.000000,-1.438455,weekday,Link Lab,2022-05-02,Monday,5.0,0.000000,9.500000,14.050000
3,edr,6,2022-05-03,Tuesday,2022-05-03 01:13:00,2022-05-03 08:22:30,25770.0,6.250000,0.315789,2100,...,0.610733,0.935394,weekday,Link Lab,2022-05-03,Tuesday,11.0,3.627500,7.050000,11.672500
4,edr,7,2022-05-04,Wednesday,2022-05-04 01:24:30,2022-05-04 08:37:30,25980.0,6.683333,0.442446,3750,...,0.663457,1.140327,weekday,Link Lab,2022-05-04,Wednesday,6.0,12.872500,8.550000,1.650000


In [50]:
# drop date_y, day_of_week
merged = merged.drop(columns=['date_x','date_y', 'day_of_week'])

## Visualizations

In [51]:
offgrounds = merged[merged['Work']=='Off Grounds']
linklab = merged[merged['Work']=='Link Lab']

In [52]:
# plot productivity vs time spent at home for all
fig = px.scatter(merged, x="time_home_hr", y="productivity", color="ParticipantID", trendline="ols")
fig.show()

/Users/beatriceli/miniconda3/envs/wellbeing/lib/python3.10/site-packages/statsmodels/regression/linear_model.py:1752: RuntimeWarning: divide by zero encountered in double_scalars
  return 1 - self.ssr/self.centered_tss


In [53]:
# plot productivity vs time spent at home for all
fig = px.scatter(merged, x="time_eschool_hr", y="productivity", color="ParticipantID", trendline="ols")
fig.show()

/Users/beatriceli/miniconda3/envs/wellbeing/lib/python3.10/site-packages/statsmodels/regression/linear_model.py:1752: RuntimeWarning:

divide by zero encountered in double_scalars



In [54]:
# plot productivity vs time spent at home for all
fig = px.scatter(linklab, x="time_eschool_hr", y="productivity", trendline="ols")
fig.show()

<!-- within-sub: compare the productivity against time in eschool and time at home -->
<!-- do the same for sleep quality -->

In [55]:
# what about for the participants that Work Off Grounds
fig = px.scatter(offgrounds, x="time_home_hr", y="productivity", color="ParticipantID", trendline="ols")
fig.show()

In [56]:
# link lab
fig = px.scatter(linklab, x="time_home_hr", y="productivity", color="ParticipantID", trendline="ols")
fig.show()

/Users/beatriceli/miniconda3/envs/wellbeing/lib/python3.10/site-packages/statsmodels/regression/linear_model.py:1752: RuntimeWarning:

divide by zero encountered in double_scalars



In [57]:
# plot productivity vs time spent outside by participant
fig = px.scatter(merged, x="time_else_hr", y="productivity", #color="ParticipantID", 
                 trendline="ols")
fig.show()

In [58]:
# plot stress vs time spent outside by participant
fig = px.scatter(merged, x="time_else_hr", y="stress", color="ParticipantID",
                    trendline="ols")
fig.show()

/Users/beatriceli/miniconda3/envs/wellbeing/lib/python3.10/site-packages/statsmodels/regression/linear_model.py:1752: RuntimeWarning:

divide by zero encountered in double_scalars



In [59]:
# plot steps vs time spent outside by participant
fig = px.scatter(merged, x="time_else_hr", y="steps", color="ParticipantID",
                    trendline="ols")
fig.show()


In [60]:
# boxplots of time_home_hr by participant where x axis is day of week
# Define the order of categories for the 'day_name' column
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

fig = px.box(merged, x='day_name', y='time_eschool_hr', color='ParticipantID',
            category_orders={'day_name': day_order})

fig.show()

In [61]:
fig = px.box(offgrounds, x='day_name', y='time_eschool_hr', color='ParticipantID',
            category_orders={'day_name': day_order})

fig.show()

In [62]:
fig = px.box(merged, x='day_name', y='time_eschool_hr', color='Work',
            category_orders={'day_name': day_order})

fig.show()

In [63]:
fig = px.box(merged, x='day_name', y='time_home_hr', color='Work',
            category_orders={'day_name': day_order})

fig.show()

In [64]:
fig = px.box(merged, x='day_name', y='productivity', color='Work',
            category_orders={'day_name': day_order})

fig.show()

In [65]:
fig = px.box(merged, x='day_name', y='stress', color='Work',
            category_orders={'day_name': day_order})

fig.show()

In [66]:
fig = px.box(merged, x='Work', y='REM:nREM', color='Work')

fig.show()

In [67]:
fig = px.box(merged, x='Work', y='sleep', color='Work')
# sleep rating

fig.show()

In [68]:
%reload_ext rpy2.ipython

In [69]:
merged.columns

Index(['ParticipantID', 'group', 'wakeDate', 'day_name', 'bedtime', 'wakeup',
       'duration', 'TST', 'REM:nREM', 'deep', 'light', 'rem', 'sleep', 'Q4',
       'rep_bedtime', 'rep_wakeup', 'day_before', 'survey_index', 'EndDate',
       'activity', 'Q29', 'stress', 'productivity', 'Q27_1', 'Q34',
       'Q34_6_TEXT', 'Q35', 'steps', 'steps_norm', 'steps_std', 'week', 'Work',
       'count_groups', 'time_else_hr', 'time_eschool_hr', 'time_home_hr'],
      dtype='object')

In [70]:
# rename REM:nREM to REMnREM
merged = merged.rename(columns={'REM:nREM':'REMnREM', 'wakeDate':'date'})

# to csv
merged.to_csv('merged.csv', index=False)

In [71]:
# %%R -o lm1
# merged <- read.csv('merged.csv')
# lm1 <- lm(productivity ~ duration + sleep + stress + steps + time_eschool_hr + time_home_hr + time_else_hr, data=merged)
# print(summary(lm1))

It seems that the stress rating and step count for the day was significant in predicting the productivity. As the coefficient for stress is higher, it seems that for each unit increase of stress, productivity will increase by more units than for each unit of steps. However, the adjusted R-sqaured is very low so it only explains a small amount of the variability in productivity so there may be other unaccounted factors that may be influential.

In [72]:
# %%R -o lm2
# lm2 <- lm(stress ~ REMnREM + sleep + productivity + steps + time_eschool_hr + time_home_hr + time_else_hr, data=merged)
# print(summary(lm2))

Sleep rating and productivity seem to be signiifcant predictors for stress but the model does not explain much variability at all.

In [73]:
# %%R
# # install.packages("lmerTest")
# library(lmerTest)
# lm3 <- lmer(productivity ~ duration + sleep + stress + steps + time_eschool_hr + time_home_hr + time_else_hr + (1|ParticipantID), data=merged)
# print(summary(lm3))

In [74]:
# %%R -o lmer1
# library(lme4)
# lmer1 <- lmer(productivity ~ REMnREM + duration + sleep + stress + steps + time_eschool_hr + time_home_hr + time_else_hr + (1|Work), data=merged)
# print(summary(lmer1))

The residual variance of 0.72967 indicates the amount of variation in productivity that is not explained by the predictor variables or the categorical variable. This means that there is something missing. **Random effects capture variability between the diffferent Work groups while fixed effects captures relationship between the predictors and outcome (productivity). None of the t-values are statistically significant under the fixed effects.

In [75]:
# %%R -o lmer2
# lmer2 <- lmer(stress ~ duration + sleep + productivity + steps + time_eschool_hr + time_home_hr + time_else_hr + (1|Work), data=merged)
# print(summary(lmer2))

## Steps (and times of getting up for Link Lab people)

In [76]:
#------------------------------------#
#----------- read step data ---------#
#------------------------------------#

all_step = []

for r,d,f in os.walk("/Users/beatriceli/Documents/GitHub/well-being/lll_fitbit_data"):
    for file in f:
        if file.endswith("steps.csv"):
            temp = pd.read_csv(os.path.join(r,file),parse_dates=["pid"])
            #temp["value"] = (temp["value"] - temp["value"].min()) 
            all_step.append(temp)

all_step = pd.concat(all_step)
all_step.columns = ["pid","Timestamp","value"]
# Convert to Datetime object
all_step['Timestamp'] = pd.to_datetime(all_step['Timestamp'], format="%Y-%m-%d %H:%M:%S")
all_step["day"] = all_step["Timestamp"].dt.day_name()
all_step["date"] = pd.to_datetime(all_step["Timestamp"]).dt.date

# remove participants n3f and iyl
all_step = all_step[all_step['pid'] != 'iyl']
all_step = all_step[all_step['pid'] != 'n3f']

all_step = all_step.sort_values(by=['pid','Timestamp'])
# rename value to steps
all_step = all_step.rename(columns = {'value':'steps'})
all_step.to_csv('all_step.csv', index=False)
all_step.head()

,pid,Timestamp,steps,day,date
0,edr,2022-04-26 00:00:00,0,Tuesday,2022-04-26
1,edr,2022-04-26 00:01:00,0,Tuesday,2022-04-26
2,edr,2022-04-26 00:02:00,0,Tuesday,2022-04-26
3,edr,2022-04-26 00:03:00,0,Tuesday,2022-04-26
4,edr,2022-04-26 00:04:00,0,Tuesday,2022-04-26


In [77]:
# create a column that checks if the steps are above 0
all_step['nonzero'] = np.where(all_step['steps']>0, True, False)
for p in pids:
    temp = all_step[all_step["pid"]==p]
    temp = temp.sort_values(by=['Timestamp'])
    temp["group"] = temp["nonzero"].ne(temp["nonzero"].shift()).cumsum()
    all_step.loc[all_step["pid"]==p,"group"] = temp["group"]

# group to int
all_step['group'] = all_step['group'].astype(int)
# drop nonzero column
all_step = all_step.drop(columns=['nonzero'])
all_step.head()

,pid,Timestamp,steps,day,date,group
0,edr,2022-04-26 00:00:00,0,Tuesday,2022-04-26,1
1,edr,2022-04-26 00:01:00,0,Tuesday,2022-04-26,1
2,edr,2022-04-26 00:02:00,0,Tuesday,2022-04-26,1
3,edr,2022-04-26 00:03:00,0,Tuesday,2022-04-26,1
4,edr,2022-04-26 00:04:00,0,Tuesday,2022-04-26,1


In [78]:
# get the first and last timestamp for each group for each participant pid and sum the steps
step_sum = all_step.groupby(['pid', 'group','day','date']).agg(
    start=('Timestamp', 'first'),
    end=('Timestamp', 'last'),
    duration=('Timestamp', lambda x: x.max() - x.min()),
    steps=('steps', 'sum')
).reset_index()

# convert duration to minutes
step_sum['duration'] = step_sum['duration'].dt.total_seconds()/60

step_sum.head()

,pid,group,day,date,start,end,duration,steps
0,edr,1,Tuesday,2022-04-26,2022-04-26 00:00:00,2022-04-26 10:52:00,652.0,0
1,edr,2,Tuesday,2022-04-26,2022-04-26 10:53:00,2022-04-26 10:53:00,0.0,8
2,edr,3,Tuesday,2022-04-26,2022-04-26 10:54:00,2022-04-26 11:19:00,25.0,0
3,edr,4,Tuesday,2022-04-26,2022-04-26 11:20:00,2022-04-26 11:20:00,0.0,9
4,edr,5,Tuesday,2022-04-26,2022-04-26 11:21:00,2022-04-26 11:23:00,2.0,0


In [79]:
# filter on participants who work at Link Lab 
ll = locsunique[locsunique['Work']=='Link Lab']
off = locsunique[locsunique['Work']=='Off Grounds']

# get at_Eschool = True in ll
ll_eschool = ll[ll['at_Eschool']==True]
# drop unecessary columns
ll_eschool = ll_eschool.drop(columns=['at_home', 'at_else', 'at_Eschool', 'Work','home_address'])
ll_eschool.head()

,ParticipantID,group,date,time_start,time_end,time_spent,latitude,longitude,unique_address,revGeo_address,check_address
0,edr,1,2022-04-26,2022-04-26 14:53:00,2022-04-26 16:08:00,4500,38.032036,-78.510156,"[""Olsson Hall, 151, Engineer's Way, McCormick ...","Olsson Hall, 151, Engineer's Way, McCormick Ro...",True
2,edr,3,2022-04-26,2022-04-26 17:17:00,2022-04-26 23:59:00,24157,38.032121,-78.510238,"['Thornton Hall, Woodrow Street, Jefferson Par...","Olsson Hall, 151, Engineer's Way, McCormick Ro...",True
3,edr,3,2022-04-27,2022-04-27 00:00:00,2022-04-27 02:03:00,7407,38.032121,-78.510238,"['Thornton Hall, Woodrow Street, Jefferson Par...","Olsson Hall, 151, Engineer's Way, McCormick Ro...",True
5,edr,5,2022-04-27,2022-04-27 12:15:00,2022-04-27 22:39:00,37440,38.032005,-78.510192,"[""Olsson Hall, 151, Engineer's Way, McCormick ...","Olsson Hall, 151, Engineer's Way, McCormick Ro...",True
10,edr,9,2022-04-28,2022-04-28 16:26:00,2022-04-28 23:59:00,27223,38.031907,-78.510663,"[""Olsson Hall, 151, Engineer's Way, McCormick ...","Olsson Hall, 151, Engineer's Way, McCormick Ro...",True


In [80]:
# create a column in step_sum that checks if start and end are between time_start and time_end for each participant, in_ll
# loop through step_sum and check if start and end are between time_start and time_end for each participant
# if true, then in_ll is true
steps_link = step_sum.copy()
steps_link['in_ll'] = False

for index, row in steps_link.iterrows():
    start = row['start']
    end = row['end']
    participant = row['pid']
    in_ll = ll_eschool[(ll_eschool['ParticipantID'] == participant) & (ll_eschool['time_start'] <= start) & (ll_eschool['time_end'] >= end)].shape[0] > 0
    steps_link.at[index, 'in_ll'] = in_ll
    
# reset index
steps_link = steps_link.reset_index(drop=True)
steps_link.head()

,pid,group,day,date,start,end,duration,steps,in_ll
0,edr,1,Tuesday,2022-04-26,2022-04-26 00:00:00,2022-04-26 10:52:00,652.0,0,False
1,edr,2,Tuesday,2022-04-26,2022-04-26 10:53:00,2022-04-26 10:53:00,0.0,8,False
2,edr,3,Tuesday,2022-04-26,2022-04-26 10:54:00,2022-04-26 11:19:00,25.0,0,False
3,edr,4,Tuesday,2022-04-26,2022-04-26 11:20:00,2022-04-26 11:20:00,0.0,9,False
4,edr,5,Tuesday,2022-04-26,2022-04-26 11:21:00,2022-04-26 11:23:00,2.0,0,False


In [81]:
# have only one timestamp column with one minute intervals between start and end
# Create a new timestamp column with one-minute intervals between start and end
steps_link['timestamp'] = steps_link.apply(lambda row: pd.date_range(start=row['start'], end=row['end'], freq='1min'), axis=1)

# Explode the rows of the DataFrame so that each row corresponds to a single timestamp
steps_link = steps_link.explode('timestamp').reset_index(drop=True)

# Calculate the steps for each timestamp based on the duration
steps_link['steps'] = steps_link.apply(lambda row: 
                                   row['steps'] if row['duration'] <= 1 else (
                                       row['steps'] / row['duration'] if row['timestamp'] < row['end'] else 0), axis=1)


# sort by pid and timestamp
steps_link = steps_link.sort_values(['pid', 'timestamp']).reset_index(drop=True)

# steps_link.head()
# Drop the start and end columns
steps_link.drop(['start', 'end','duration'], axis=1, inplace=True)
# # reorder columns
steps_link = steps_link[['pid', 'group', 'day', 'date', 'timestamp', 'steps', 'in_ll']]

# flag if there is a break in timestamp aka not wearing watch
# if there is a break in data, then flag it as true, if there is no break in data, then flag it as false
# create a column break
steps_link['break'] = False
for p in pids:
    temp = steps_link[steps_link["pid"]==p]
    temp = temp.sort_values(by=['timestamp'])
    temp["break"] = temp["timestamp"].diff() > pd.Timedelta(hours=1)
    steps_link.loc[steps_link["pid"]==p,"break"] = temp["break"]

# Save the steps_link dataframe to a new CSV file
steps_link.to_csv('steps_ll.csv', index=False)

# print how many in_ll
print(steps_link['in_ll'].value_counts())
print(steps_link['break'].value_counts())
# print how many participants
print(steps_link['pid'].nunique())

False    1356780
True       84660
Name: in_ll, dtype: int64
False    1441440
Name: break, dtype: int64
13


In [83]:
# get at_home = True in off
off_home = off[off['at_home']==True]
# drop unecessary columns
off_home = off_home.drop(columns=['at_home', 'at_else', 'at_Eschool', 'Work','home_address'])
off_home.head()

,ParticipantID,group,date,time_start,time_end,time_spent,latitude,longitude,unique_address,revGeo_address,check_address
1722,h9u,4,2022-05-05,2022-05-05 20:19:00,2022-05-05 20:28:00,540,38.018378,-78.498662,"['100, Hurst Lane, Johnson Village, West View ...","100, Hurst Lane, Johnson Village, West View Te...",True
1724,h9u,6,2022-05-05,2022-05-05 21:04:00,2022-05-05 22:13:00,4140,38.018288,-78.498755,"['104, Hurst Lane, Johnson Village, West View ...","100, Hurst Lane, Johnson Village, West View Te...",True
1727,h9u,8,2022-05-06,2022-05-06 15:31:00,2022-05-06 15:40:00,540,38.018346,-78.498682,"['100, Hurst Lane, Johnson Village, West View ...","100, Hurst Lane, Johnson Village, West View Te...",True
1739,h9u,17,2022-05-10,2022-05-10 17:16:00,2022-05-10 17:19:00,180,38.018333,-78.498623,"['100, Hurst Lane, Johnson Village, West View ...","100, Hurst Lane, Johnson Village, West View Te...",True
1764,h9u,41,2022-05-11,2022-05-11 23:10:00,2022-05-11 23:28:00,1080,38.018374,-78.498714,"['100, Hurst Lane, Johnson Village, West View ...","100, Hurst Lane, Johnson Village, West View Te...",True


In [84]:
steps_off = step_sum.copy()
steps_off['in_off'] = False

for index, row in steps_off.iterrows():
    start = row['start']
    end = row['end']
    participant = row['pid']
    in_off = off_home[(off_home['ParticipantID'] == participant) & (off_home['time_start'] <= start) & (off_home['time_end'] >= end)].shape[0] > 0
    steps_off.at[index, 'in_off'] = in_off

# reset index
steps_off = steps_off.reset_index(drop=True)
steps_off.head()

,pid,group,day,date,start,end,duration,steps,in_off
0,edr,1,Tuesday,2022-04-26,2022-04-26 00:00:00,2022-04-26 10:52:00,652.0,0,False
1,edr,2,Tuesday,2022-04-26,2022-04-26 10:53:00,2022-04-26 10:53:00,0.0,8,False
2,edr,3,Tuesday,2022-04-26,2022-04-26 10:54:00,2022-04-26 11:19:00,25.0,0,False
3,edr,4,Tuesday,2022-04-26,2022-04-26 11:20:00,2022-04-26 11:20:00,0.0,9,False
4,edr,5,Tuesday,2022-04-26,2022-04-26 11:21:00,2022-04-26 11:23:00,2.0,0,False


In [85]:
steps_off['timestamp'] = steps_off.apply(lambda row: pd.date_range(start=row['start'], end=row['end'], freq='1min'), axis=1)

# Explode the rows of the DataFrame so that each row corresponds to a single timestamp
steps_off = steps_off.explode('timestamp').reset_index(drop=True)

# Calculate the steps for each timestamp based on the duration
steps_off['steps'] = steps_off.apply(lambda row:
                                      row['steps'] if row['duration'] <= 1 else (
                                            row['steps'] / row['duration'] if row['timestamp'] < row['end'] else 0), axis=1)

# sort by pid and timestamp
steps_off = steps_off.sort_values(['pid', 'timestamp']).reset_index(drop=True)

# Drop the start and end columns
steps_off.drop(['start', 'end','duration'], axis=1, inplace=True)
# reorder columns
steps_off = steps_off[['pid', 'group', 'day', 'date', 'timestamp', 'steps', 'in_off']]

# flag if there is a break in timestamp aka not wearing watch
# if there is a break in data, then flag it as true, if there is no break in data, then flag it as false
# create a column break
steps_off['break'] = False
for p in pids:
    temp = steps_off[steps_off["pid"]==p]
    temp = temp.sort_values(by=['timestamp'])
    temp["break"] = temp["timestamp"].diff() > pd.Timedelta(hours=1)
    steps_off.loc[steps_off["pid"]==p,"break"] = temp["break"]

# Save the steps_off dataframe to a new CSV file
steps_off.to_csv('steps_off.csv', index=False)

# print how many in_off
print(steps_off['in_off'].value_counts())
print(steps_off['break'].value_counts())
# print how many participants
print(steps_off['pid'].nunique())

False    1366847
True       74593
Name: in_off, dtype: int64
False    1441440
Name: break, dtype: int64
13


In [ ]:
# # count how many rows in step_sum that are within the start and end time of ll_eschool for each participant
# ll_eschool['steps'] = 0
# ll_eschool['gettingup'] = 0
# for index, row in ll_eschool.iterrows():
#     temp = all_step[(all_step['pid']==row['ParticipantID']) & (all_step['Timestamp']>=row['time_start']) & (all_step['Timestamp']<=row['time_end'])]
#     # count how many unique groups there are
#     ll_eschool.loc[index,'gettingup'] = temp['group'].nunique()
#     ll_eschool.loc[index,'steps'] = temp['steps'].sum().astype(int)
# # to_csv
# ll_eschool.to_csv('ll_eschool.csv', index=False)
# ll_eschool.head()

#### Daily summary of steps and count of getting up for each participant and date

In [ ]:
# groupby pid and date, sum steps and last count of unique group
daily_step = all_step.groupby(['pid','date']).agg({'steps': 'sum', 'group': 'nunique'}).reset_index()
daily_step = daily_step.rename(columns = {'group':'gettingup', 'pid':'ParticipantID'})
daily_step.head()

In [ ]:
# locsunique['steps'] = 0
# locsunique['gettingup'] = 0
# for index, row in locsunique.iterrows():
#     temp = all_step[(all_step['pid']==row['ParticipantID']) & (all_step['Timestamp']>=row['time_start']) & (all_step['Timestamp']<=row['time_end'])]
#     # count how many unique groups there are
#     locsunique.loc[index,'gettingup'] = temp['group'].nunique()
#     locsunique.loc[index,'steps'] = temp['steps'].sum().astype(int)

# # drop latitude, longitude, unique_address, home_address
# loc2 = locsunique.drop(columns=['latitude','longitude','unique_address','home_address'])
# loc2.head()

In [ ]:
# # group by ParticipantID and date, get steps, gettingup,
# daily = loc2.groupby(['ParticipantID','date']).agg({'steps':'sum','gettingup':'sum'}).reset_index()
# daily.head()

In [ ]:
# merge daily with merged on ParticipantID and date
# convert to date in both dataframes
merged['day_before'] = pd.to_datetime(merged['day_before'])
daily_step['date'] = pd.to_datetime(daily_step['date'])
# merge on day_before (merged) and date (daily_step)
merged2 = pd.merge(merged, daily_step, how='left', left_on=['ParticipantID','day_before'], right_on=['ParticipantID','date'])
# remove date_y, steps_y
merged2 = merged2.drop(columns=['date_y', 'steps_y'])
# rename date_x, steps_x
merged2 = merged2.rename(columns={'date_x':'date', 'steps_x':'steps'})
merged2.head()

In [ ]:
# plot gettingup vs productivity
fig = px.scatter(merged2, x='gettingup', y='productivity', color='ParticipantID', trendline='ols')
fig.show()

In [ ]:
# plot gettingup vs stress
fig = px.scatter(merged2, x='gettingup', y='stress', color='Work', trendline='ols')
fig.show()

## Screen time?

In [ ]:
# read in screen_use.csv
screen = pd.read_csv('screen_status.csv')
# convert to datetime
screen['date'] = pd.to_datetime(screen['date'])
# drop Work
screen = screen.drop(columns=['Work'])
screen.head()

In [ ]:
# merge with merged2 on ParticipantID and date
mergedscreen = pd.merge(merged2, screen, how='left', on=['ParticipantID','date'])
mergedscreen.head()

In [ ]:
# visualize screen_use vs productivity 
# on vs unlocked
fig = px.scatter(mergedscreen, x='sum_unlocked', y='productivity', color='Work', trendline='ols')
fig.show()

In [ ]:
# ParticipantID that works Off Grounds
off = mergedscreen[mergedscreen['Work']=='Off Grounds']
off['ParticipantID'].unique()


In [ ]:
# unlocked_duration vs TST
fig = px.scatter(mergedscreen, x='sum_unlocked', y='TST', color='Work', trendline='ols')
fig.show()

In [ ]:
# unlocked_duration vs TST
fig = px.scatter(mergedscreen, x='sum_unlocked', y='REMnREM', color='Work', trendline='ols')
fig.show()

In [ ]:
# unlocked_duration vs TST
fig = px.scatter(mergedscreen, x='sum_unlocked', y='sleep', color='Work', trendline='ols')
fig.show()

In [ ]:
# count_unlocked vs productivity
fig = px.scatter(mergedscreen, x='count_unlocked', y='productivity', color='Work', trendline='ols')
fig.show()

## Calls?

In [ ]:
# read in aware_calls.csv
calls = pd.read_csv('aware_calls.csv')
# convert date to datetime
calls['date'] = pd.to_datetime(calls['date'])
# drop Work
calls = calls.drop(columns=['Work'])
calls.head()

There is no sum duration for incoming and dialing calls as those durations are all 0.

In [ ]:
# merge with mergedscreen on ParticipantID and date
mergedphone = pd.merge(mergedscreen, calls, how='left', on=['ParticipantID','date'])
mergedphone.head()

In [ ]:
# plot count_incoming vs productivity
fig = px.scatter(mergedphone, x='count_incoming', y='productivity', color='Work', trendline='ols')
fig.show()

In [ ]:
# plot count_connected vs productivity
fig = px.scatter(mergedphone, x='count_connected', y='productivity', color='ParticipantID', trendline='ols')
fig.show()

In [ ]:
# plot sum_connected vs productivity
fig = px.scatter(mergedphone, x='sum_connected', y='productivity', color='ParticipantID', trendline='ols')
fig.show()

In [ ]:
# plot sum_disconnected vs productivity
fig = px.scatter(mergedphone, x='sum_disconnected', y='productivity', color='Work', trendline='ols')
fig.show()

In [ ]:
mergedphone.columns

In [ ]:
# to csv
mergedphone.to_csv('mergedphone.csv', index=False)

In [ ]:
%%R -o lmer3
library(lme4)
mergedphone <- read.csv('mergedphone.csv')
lmer3 <- lmer(productivity ~ REMnREM + duration + sleep + stress + steps + time_eschool_hr + time_home_hr + time_else_hr + gettingup + count_locked + count_unlocked + sum_locked +
       sum_unlocked + count_connected + count_dialing + count_disconnected + count_incoming + sum_connected + sum_disconnected + (1|Work), data=mergedphone)
print(summary(lmer3))

# Questions

### Do participants who spend more time at Eschool or at home have different sleep patterns (e.g., earlier bedtimes or wake times) compared to those who spend more time elsewhere?

In [ ]:
merged.head()

In [ ]:
# calculate average time spent at home, at Eschool, and elsewhere by per day per participant
# group by ParticipantID and day, and get the mean bedtime, wakeup, sleep duration, TST, REM:nREM, time_home_hr, time_eschool_hr, time_else_hr
ptime = merged.groupby(['ParticipantID','Work','day_name']).agg(
    # bedtime=('bedtime', 'mean'),
    # wakeup=('wakeup', 'mean'),
    sleep_duration=('duration', 'mean'),
    TST=('TST', 'mean'),
    REMnREM=('REMnREM', 'mean'),
    time_home_hr=('time_home_hr', 'mean'),
    time_eschool_hr=('time_eschool_hr', 'mean'),
    time_else_hr=('time_else_hr', 'mean'),
    productivity=('productivity', 'mean'),
    stress=('stress', 'mean'),
    steps=('steps', 'mean')
).reset_index()

ptime
# # convert bedtime and wakeup to datetime objects
# ptime['bedtime'] = pd.to_datetime(ptime['bedtime'], unit='ns')  
# ptime['wakeup'] = pd.to_datetime(ptime['wakeup'], unit='ns')

In [ ]:
ptime2 = merged.groupby(['ParticipantID','Work']).agg(
    # bedtime=('bedtime', 'mean'),
    # wakeup=('wakeup', 'mean'),
    sleep_duration=('duration', 'mean'),
    TST=('TST', 'mean'),
    REMnREM=('REMnREM', 'mean'),
    time_home_hr=('time_home_hr', 'mean'),
    time_eschool_hr=('time_eschool_hr', 'mean'),
    time_else_hr=('time_else_hr', 'mean'),
    productivity=('productivity', 'mean'),
    stress=('stress', 'mean'),
    steps=('steps', 'mean')
).reset_index()

ptime2